In [1]:
# Celda 1 — Imports y carga del último JSON EVR

import json
import glob
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)

# IMPORTANTE: carpeta base, no el fichero
DATAPATH = Path("../..") / "data" / "evr"
print("Ruta de datos EVR:", DATAPATH.resolve())

# Buscar recursivamente en cualquier subcarpeta año/mes
archivos = sorted(glob.glob(str(DATAPATH / "**" / "*.json"), recursive=True))
print("JSONs EVR encontrados:", len(archivos))
for f in archivos[-5:]:
    print(" ", f)

if not archivos:
    raise FileNotFoundError("No hay JSONs en data/evr todavía. Lanza scheduler.py primero.")

# Cargar el más reciente
f_ultimo = archivos[-1]
with open(f_ultimo, "r", encoding="utf-8") as f:
    data = json.load(f)

print("\nJSON EVR cargado:", f_ultimo)
print("Timestamp:", data.get("timestamp"))
print("Claves disponibles:", list(data.keys()))


Ruta de datos EVR: /workspaces/TFG_Spain-s_Migratory_Flow/data/evr
JSONs EVR encontrados: 2
  ../../data/evr/2026/03/26_03_2026.json
  ../../data/evr/2026/03/27_03_2026.json

JSON EVR cargado: ../../data/evr/2026/03/27_03_2026.json
Timestamp: 2026-03-27T12:46:30.755259
Claves disponibles: ['timestamp', 'flujo_interprovincial', 'saldo_interprovincial', 'flujo_edad_sexo']


In [2]:
# Celda 2 — Explorar estructura bruta EVR (flujo, saldo, edad-sexo)

for clave in ["flujo_interprovincial", "saldo_interprovincial", "flujo_edad_sexo"]:
    bloque = data.get(clave, {})
    print(f"\n=== {clave} ===")
    if "error" in bloque:
        print("  ❌ Error:", bloque["error"])
        continue

    print("  Tabla INE id:", bloque.get("tabla_id"))
    print("  Series:", bloque.get("series"))
    print("  Registros planos:", bloque.get("registros"))

    muestras = bloque.get("data", [])[:3]
    if not muestras:
        print("  (sin datos)")
        continue

    print("  Ejemplo de 'nombre_serie':", muestras[0].get("nombre_serie"))
    print("  Ejemplo de registro Data:", muestras[0])



=== flujo_interprovincial ===
  Tabla INE id: 24379
  Series: 8427
  Registros planos: 115148
  Ejemplo de 'nombre_serie': Total. Total Nacional. Flujo de migraciones interprovinciales. Total Nacional. 
  Ejemplo de registro Data: {'nombre_serie': 'Total. Total Nacional. Flujo de migraciones interprovinciales. Total Nacional. ', 'anyo': 2021, 'valor': 482346.0, 'secreto': False}

=== saldo_interprovincial ===
  Tabla INE id: 24380
  Series: 14628
  Registros planos: 200741
  Ejemplo de 'nombre_serie': Total. Albacete. Saldo de migración interprovincial. Todas las edades. 
  Ejemplo de registro Data: {'nombre_serie': 'Total. Albacete. Saldo de migración interprovincial. Todas las edades. ', 'anyo': 2021, 'valor': -387.0, 'secreto': False}

=== flujo_edad_sexo ===
  Tabla INE id: 24339
  Series: 276
  Registros planos: 3864
  Ejemplo de 'nombre_serie': Total Nacional. Total. Flujo de migraciones interprovinciales. Todas las edades. 
  Ejemplo de registro Data: {'nombre_serie': 'Total Na

In [3]:
# Celda 3 — Función para extraer dimensiones de nombre_serie

def parse_dims_flujo(nombre: str) -> dict:
    """
    Para tabla 24379 (flujo interprovincial):
    Ejemplo de Nombre en INE: 
    'Total. Total Nacional. Flujo de migraciones interprovinciales. Total Nacional.'
    o 'Total. Madrid, Comunidad de. Flujo de migraciones interprovinciales. Barcelona.'
    La estructura útil es:
      [sexo] . [provincia_origen] . 'Flujo de migraciones interprovinciales.' . [provincia_destino]
    Simplificamos parsing por posiciones.
    """
    partes = [p.strip() for p in nombre.split(".") if p.strip()]
    # fallback seguro
    dims = {
        "sexo": None,
        "provincia_origen": None,
        "provincia_destino": None,
    }
    if len(partes) >= 4:
        dims["sexo"] = partes[0]
        # En muchas series, partes[1] = origen, partes[-1] = destino
        dims["provincia_origen"] = partes[1]
        dims["provincia_destino"] = partes[-1]
    elif len(partes) == 3:
        dims["sexo"] = partes[0]
        dims["provincia_origen"] = partes[1]
        dims["provincia_destino"] = partes[2]
    else:
        # Total nacional u otros casos raros
        dims["sexo"] = partes[0] if partes else None

    return dims


def parse_dims_saldo(nombre: str) -> dict:
    """
    Para tabla 24380 (saldo interprovincial):
    Estructura tipo:
      'Total. Saldo de migración interprovincial. Madrid, Comunidad de. Todas las edades.'
    Aproximación:
      partes[0] = sexo (Total / Hombres / Mujeres)
      alguna parte contiene la provincia
      última parte suele ser grupo de edad ('Todas las edades', 'De 0 a 4 años', etc.)
    """
    partes = [p.strip() for p in nombre.split(".") if p.strip()]
    dims = {
        "sexo": None,
        "provincia": None,
        "grupo_edad": None,
    }
    if not partes:
        return dims

    dims["sexo"] = partes[0]

    if len(partes) >= 3:
        dims["provincia"] = partes[2]  # suele ser la provincia
    if len(partes) >= 4:
        dims["grupo_edad"] = partes[-1]

    return dims


In [4]:
# Celda 4 — Parsear flujo interprovincial provincia_origen×destino×año×sexo

flujo_raw = data["flujo_interprovincial"]["data"]
print("Registros brutos flujo_interprovincial:", len(flujo_raw))

filas = []
for r in flujo_raw:
    dims = parse_dims_flujo(r["nombre_serie"])
    filas.append(
        {
            "sexo": dims["sexo"],
            "provincia_origen": dims["provincia_origen"],
            "provincia_destino": dims["provincia_destino"],
            "anyo": r["anyo"],
            "valor": r["valor"],
            "secreto": r["secreto"],
        }
    )

df_flujo = pd.DataFrame(filas)
print("Shape flujo interprovincial:", df_flujo.shape)
print("Columnas:", list(df_flujo.columns))
print("\nMuestra:")
print(df_flujo.head(10).to_string(index=False))

print("\nAños disponibles:", sorted(df_flujo["anyo"].dropna().unique().tolist()))
print("Provincias origen:", df_flujo["provincia_origen"].dropna().unique()[:10])
print("Provincias destino:", df_flujo["provincia_destino"].dropna().unique()[:10])


Registros brutos flujo_interprovincial: 115148
Shape flujo interprovincial: (115148, 6)
Columnas: ['sexo', 'provincia_origen', 'provincia_destino', 'anyo', 'valor', 'secreto']

Muestra:
 sexo provincia_origen provincia_destino  anyo    valor  secreto
Total   Total Nacional    Total Nacional  2021 482346.0    False
Total   Total Nacional    Total Nacional  2020 416902.0    False
Total   Total Nacional    Total Nacional  2019 475859.0    False
Total   Total Nacional    Total Nacional  2018 465536.0    False
Total   Total Nacional    Total Nacional  2017 443729.0    False
Total   Total Nacional    Total Nacional  2016 448418.0    False
Total   Total Nacional    Total Nacional  2015 465514.0    False
Total   Total Nacional    Total Nacional  2014 464142.0    False
Total   Total Nacional    Total Nacional  2013 467396.0    False
Total   Total Nacional    Total Nacional  2012 483793.0    False

Años disponibles: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2

In [5]:
# Celda 5 — Parsear saldo interprovincial provincia×año×sexo×grupo_edad

saldo_raw = data["saldo_interprovincial"]["data"]
print("Registros brutos saldo_interprovincial:", len(saldo_raw))

filas = []
for r in saldo_raw:
    dims = parse_dims_saldo(r["nombre_serie"])
    filas.append(
        {
            "sexo": dims["sexo"],
            "provincia": dims["provincia"],
            "grupo_edad": dims["grupo_edad"],
            "anyo": r["anyo"],
            "valor": r["valor"],
            "secreto": r["secreto"],
        }
    )

df_saldo = pd.DataFrame(filas)
print("Shape saldo interprovincial:", df_saldo.shape)
print("Columnas:", list(df_saldo.columns))
print("\nMuestra:")
print(df_saldo.head(10).to_string(index=False))

print("\nAños disponibles:", sorted(df_saldo["anyo"].dropna().unique().tolist()))
print("Provincias:", df_saldo["provincia"].dropna().unique()[:10])
print("Grupos de edad:", df_saldo["grupo_edad"].dropna().unique()[:10])


Registros brutos saldo_interprovincial: 200741
Shape saldo interprovincial: (200741, 6)
Columnas: ['sexo', 'provincia', 'grupo_edad', 'anyo', 'valor', 'secreto']

Muestra:
 sexo                          provincia       grupo_edad  anyo   valor  secreto
Total Saldo de migración interprovincial Todas las edades  2021  -387.0    False
Total Saldo de migración interprovincial Todas las edades  2020   -30.0    False
Total Saldo de migración interprovincial Todas las edades  2019  -765.0    False
Total Saldo de migración interprovincial Todas las edades  2018  -882.0    False
Total Saldo de migración interprovincial Todas las edades  2017 -1202.0    False
Total Saldo de migración interprovincial Todas las edades  2016 -1391.0    False
Total Saldo de migración interprovincial Todas las edades  2015 -1502.0    False
Total Saldo de migración interprovincial Todas las edades  2014 -1356.0    False
Total Saldo de migración interprovincial Todas las edades  2013 -1408.0    False
Total Saldo de mig

In [6]:
# Celda 6 — Parsear flujo interprovincial nacional×año×sexo×edad

edad_raw = data["flujo_edad_sexo"]["data"]
print("Registros brutos flujo_edad_sexo:", len(edad_raw))

# Para esta tabla el Nombre suele llevar sexo y grupo de edad, pero sin provincias
def parse_dims_edad(nombre: str) -> dict:
    partes = [p.strip() for p in nombre.split(".") if p.strip()]
    dims = {"sexo": None, "grupo_edad": None}
    if partes:
        dims["sexo"] = partes[0]
    if len(partes) >= 2:
        dims["grupo_edad"] = partes[-1]
    return dims

filas = []
for r in edad_raw:
    dims = parse_dims_edad(r["nombre_serie"])
    filas.append(
        {
            "sexo": dims["sexo"],
            "grupo_edad": dims["grupo_edad"],
            "anyo": r["anyo"],
            "valor": r["valor"],
            "secreto": r["secreto"],
        }
    )

df_edad = pd.DataFrame(filas)
print("Shape flujo_edad_sexo:", df_edad.shape)
print("Columnas:", list(df_edad.columns))
print("\nMuestra:")
print(df_edad.head(10).to_string(index=False))

print("\nAños disponibles:", sorted(df_edad["anyo"].dropna().unique().tolist()))
print("Grupos de edad:", df_edad["grupo_edad"].dropna().unique()[:10])


Registros brutos flujo_edad_sexo: 3864
Shape flujo_edad_sexo: (3864, 5)
Columnas: ['sexo', 'grupo_edad', 'anyo', 'valor', 'secreto']

Muestra:
          sexo       grupo_edad  anyo    valor  secreto
Total Nacional Todas las edades  2021 482346.0    False
Total Nacional Todas las edades  2020 416902.0    False
Total Nacional Todas las edades  2019 475859.0    False
Total Nacional Todas las edades  2018 465536.0    False
Total Nacional Todas las edades  2017 443729.0    False
Total Nacional Todas las edades  2016 448418.0    False
Total Nacional Todas las edades  2015 465514.0    False
Total Nacional Todas las edades  2014 464142.0    False
Total Nacional Todas las edades  2013 467396.0    False
Total Nacional Todas las edades  2012 483793.0    False

Años disponibles: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
Grupos de edad: <ArrowStringArray>
['Todas las edades',           '0 años',            '1 año',
           '2 años',           '3 años', 

In [7]:
# Celda 7 — Resumen rápido de calidad EVR

def resumen_calidad(nombre, df, cols_num):
    if df.empty:
        print(f"{nombre}: DataFrame vacío")
        return
    total_celdas = df[cols_num].size
    nulos_total = df[cols_num].isnull().sum().sum()
    pct_nulos = round(nulos_total / total_celdas * 100, 1)
    print(f"{nombre}")
    print(f"  Filas: {len(df)} | Columnas: {len(df.columns)} | Nulos: {pct_nulos}%")
    nulos_col = df[cols_num].isnull().sum()
    nulos_col = nulos_col[nulos_col > 0]
    if len(nulos_col):
        print("  Nulos por columna:")
        for c, n in nulos_col.items():
            print(f"    {c}: {n} ({n/len(df)*100:.1f}%)")
    print()

resumen_calidad("Flujo interprovincial", df_flujo, ["valor"])
resumen_calidad("Saldo interprovincial", df_saldo, ["valor"])
resumen_calidad("Flujo edad-sexo", df_edad, ["valor"])


Flujo interprovincial
  Filas: 115148 | Columnas: 6 | Nulos: 0.0%

Saldo interprovincial
  Filas: 200741 | Columnas: 6 | Nulos: 0.0%

Flujo edad-sexo
  Filas: 3864 | Columnas: 5 | Nulos: 0.0%



In [8]:
# Celda 8 — Guardar parquets RAW EVR para fase 02_limpieza

OUTPUT_DIR = Path("../..") / "output" / "evr" / "01-raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

outputs = {
    "raw_flujo_interprovincial.parquet": df_flujo,
    "raw_saldo_interprovincial.parquet": df_saldo,
    "raw_flujo_nacional_edad_sexo.parquet": df_edad,
}

for fname, df in outputs.items():
    path = OUTPUT_DIR / fname
    df.to_parquet(path, index=False)
    print(f"✅ Guardado: {path} ({len(df)} filas)")

print("\n🎉 Todos los parquets EVR RAW guardados en output/evr/01-raw")


✅ Guardado: ../../output/evr/01-raw/raw_flujo_interprovincial.parquet (115148 filas)
✅ Guardado: ../../output/evr/01-raw/raw_saldo_interprovincial.parquet (200741 filas)
✅ Guardado: ../../output/evr/01-raw/raw_flujo_nacional_edad_sexo.parquet (3864 filas)

🎉 Todos los parquets EVR RAW guardados en output/evr/01-raw
